# ML Assignment 2 – Bank Marketing Classification

**Goal:** predict whether a client subscribes to a term deposit (`y`), comparing five classifiers with six metrics.

**Dataset:** UCI Bank Marketing, `bank-additional-full.csv`

## Contents
1. Setup and load data
2. Data understanding
3. Preprocessing and models *(later stages)*

## 1. Setup and load data

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

print('pandas', pd.__version__, '| numpy', np.__version__, '| scikit-learn', sklearn.__version__)

pandas 3.0.5 | numpy 2.5.3 | scikit-learn 1.9.1


In [2]:
# The file is semicolon-separated
data = pd.read_csv('data/bank-additional-full.csv', sep=';')
print('Shape:', data.shape)
data.head()

Shape: (41188, 21)


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


## 2. Data understanding

### 2.1 Structure, data types and summary statistics

In [ ]:
data.info()

In [ ]:
data.describe().T.round(2)

### 2.2 Missing values and duplicates

In [ ]:
print('Null values per column:')
print(data.isnull().sum()[data.isnull().sum() > 0] if data.isnull().sum().sum() else 'None - no null values')
print('\nDuplicate rows:', data.duplicated().sum())

In [ ]:
# Categorical columns use the string 'unknown' as a missing-value marker
cat_cols = data.drop(columns='y').select_dtypes(exclude='number').columns.tolist()
unknown_counts = (data[cat_cols] == 'unknown').sum()
unknown_counts[unknown_counts > 0].sort_values(ascending=False).to_frame('unknown_count').assign(
    pct=lambda d: (d.unknown_count / len(data) * 100).round(2))

**Findings**
- There are no real `NaN` values, but several categorical columns hold `'unknown'` (largest share: `default`). These will be treated as their own category rather than dropped.
- There are a small number of exact duplicate rows; they will be removed before splitting (next stage) so the same record cannot appear in both train and test.

### 2.3 Target variable (`y`)

In [ ]:
counts = data['y'].value_counts()
pct = data['y'].value_counts(normalize=True).mul(100).round(2)
print(pd.DataFrame({'count': counts, 'percent': pct}))

fig, ax = plt.subplots(figsize=(4, 3))
sns.countplot(data=data, x='y', ax=ax)
ax.set_title('Target distribution')
plt.show()

**Finding:** the target is imbalanced (about 11% `yes`). Accuracy alone would be misleading - a model that always predicts `no` scores about 89%. Precision, recall, F1, AUC and especially MCC will be needed to judge the models, and the train/test split must be **stratified**.

### 2.4 Numerical and categorical predictors

In [ ]:
X_all = data.drop(columns='y')
num_cols = X_all.select_dtypes(include='number').columns.tolist()
cat_cols = X_all.select_dtypes(exclude='number').columns.tolist()
print(f'Numerical predictors ({len(num_cols)}):', num_cols)
print(f'Categorical predictors ({len(cat_cols)}):', cat_cols)
print('Total predictors:', X_all.shape[1])

In [ ]:
data[cat_cols].nunique().sort_values(ascending=False).to_frame('unique_values')

### 2.5 Distributions and relationships

In [ ]:
data[num_cols].hist(bins=30, figsize=(14, 8))
plt.tight_layout()
plt.show()

In [ ]:
# Subscription rate by category for a few key categorical features
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col in zip(axes, ['job', 'poutcome', 'month']):
    rate = data.groupby(col)['y'].apply(lambda s: (s == 'yes').mean() * 100).sort_values()
    rate.plot(kind='barh', ax=ax)
    ax.set_title(f'% subscribed by {col}')
    ax.set_xlabel('% yes')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9, 7))
sns.heatmap(data[num_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation of numerical predictors')
plt.show()

**Findings**
- `pdays` uses the code `999` for 'client not previously contacted', so it is not a true numeric distance and is highly skewed.
- `campaign`, `previous` and `duration` are right-skewed; scaling will matter for Logistic Regression, KNN and SVM.
- Macro-economic features (`emp.var.rate`, `euribor3m`, `nr.employed`) are strongly correlated with each other (multicollinearity), which is acceptable for tree models but worth noting for Logistic Regression.
- `duration` (call length) is only known after the call ends, so it would not be available for a real pre-call prediction. It is kept here because it is part of the assignment dataset, and this limitation will be stated in the README.